# Minute-Level Future Direction Baseline (Notebook)

This notebook prepares a baseline pipeline for predicting the next-minute direction of a major futures contract. It follows the provided specification: feature construction, label generation, rolling-window cross-validation, class-weighted linear models, standardized features, and reusable visualization helpers. All comments and printed text are in English to avoid font issues, while figure titles and labels are also in English.

In [ ]:
import os
from pathlib import Path
from typing import List, Tuple, Dict, Optional
from datetime import datetime

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, precision_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')

# Configure pandas display for easier debugging inside the notebook
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)



## Paths and lightweight configuration

Large data (8-9 GB) lives in the folder `2005年__20250905`. To avoid long runs, set the following knobs:

* `DATA_DIR`: root folder containing CSVs.
* `MAX_FILES`: limit how many CSVs to process (None means all, but keep it small for quick tests).
* `NROWS_PER_FILE`: optional cap on rows per file while prototyping.
* `START_DATE` / `END_DATE`: limit the date interval (inclusive) used after loading.

You can raise these limits later when running the full dataset.

In [ ]:
# Root folder for CSV files
DATA_DIR = Path('2005年__20250905')

# Control how many files to load (set to None to load all files, but that can be very slow)
MAX_FILES: Optional[int] = 1

# Optional cap on rows per file for quick experiments (set to None to load full file)
NROWS_PER_FILE: Optional[int] = None 

# Time window filter (inclusive). Use None to skip filtering.
START_DATE: Optional[str] = '2005-01-01'
END_DATE: Optional[str] = '2013-12-31'

# Time-stamped output folders for each run
RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_DIR = Path('outputs')
RUN_DIR = OUTPUT_DIR / RUN_TIMESTAMP
FIG_DIR = RUN_DIR / 'figures'
METRIC_DIR = RUN_DIR / 'metrics'
for path in [FIG_DIR, METRIC_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(f'Data directory: {DATA_DIR.resolve()}')
print(f'Run timestamp: {RUN_TIMESTAMP}')
print(f'Figures will be saved under: {FIG_DIR.resolve()}')
print(f'Metrics will be saved under: {METRIC_DIR.resolve()}')



## Utility: list CSV files and pick one

We first list available CSVs to choose a subset. Because there may be many files, only the first few names are shown. Adjust `MAX_FILES` above to control how many files will be loaded.

In [ ]:
def list_csv_files(data_dir: Path, limit: Optional[int] = None) -> List[Path]:
    '''Return a sorted list of CSV files under `data_dir`, optionally limited.'''
    files = sorted(data_dir.glob('*.csv'))
    if limit is not None:
        files = files[:limit]
    print(f"Found {len(files)} CSV files (showing up to {limit}):")
    for name in files[:5]:
        print(f"  - {name.name}")
    return files

csv_files = list_csv_files(DATA_DIR, limit=MAX_FILES)
if not csv_files:
    print("No CSV files were found. Please check DATA_DIR.")


## Data loading helper

Loads one CSV at a time with optional row and date filters to keep exploratory runs fast. Progress messages are printed to help monitor long reads.

In [ ]:
def load_single_csv(path: Path, start_date: Optional[str] = None, end_date: Optional[str] = None, nrows: Optional[int] = None) -> pd.DataFrame:
    '''Load a single CSV with optional row and date filtering.'''
    print(f"Loading file: {path.name}")
    df = pd.read_csv(
        path,
        parse_dates=['index'],
        nrows=nrows,
    )
    print(f"Loaded shape before filtering: {df.shape}")

    if start_date is not None:
        df = df[df['index'] >= pd.to_datetime(start_date)]
    if end_date is not None:
        df = df[df['index'] <= pd.to_datetime(end_date)]

    df = df.sort_values('index').reset_index(drop=True)
    print(f"Shape after sorting and date filter: {df.shape}")
    return df


def load_dataset(files: List[Path], start_date: Optional[str], end_date: Optional[str], nrows_per_file: Optional[int]) -> pd.DataFrame:
    '''Concatenate multiple CSV files while applying the same filters.'''
    frames = []
    for idx, path in enumerate(files, start=1):
        print(f"
[Progress] Loading file {idx}/{len(files)}")
        frames.append(load_single_csv(path, start_date=start_date, end_date=end_date, nrows=nrows_per_file))
    if not frames:
        raise ValueError("No data was loaded; please check file list and filters.")
    data = pd.concat(frames, ignore_index=True)
    print(f"Total concatenated shape: {data.shape}")
    return data

if csv_files:
    data_raw = load_dataset(csv_files, START_DATE, END_DATE, NROWS_PER_FILE)
else:
    data_raw = pd.DataFrame()


## Feature engineering

Implements all baseline features described in the specification. Each step adds new columns; missing values from lags or rolling windows are handled later by dropping the earliest rows.

In [ ]:
def compute_return(df: pd.DataFrame) -> pd.Series:
    return np.log(df['close']).diff()


def add_feature_columns(df: pd.DataFrame, lag: int = 5, rsi_window: int = 5, range_window: int = 5) -> pd.DataFrame:
    '''Add baseline features to the provided DataFrame.'''
    df = df.copy()

    # Minute log returns and lags
    df['r_t'] = compute_return(df)
    for k in range(1, lag + 1):
        df[f'r_t_minus_{k}'] = df['r_t'].shift(k)

    # Moving average of close
    df[f'ma_{rsi_window}'] = df['close'].rolling(rsi_window).mean()

    # RSI components
    delta_close = df['close'].diff()
    gain = delta_close.clip(lower=0)
    loss = (-delta_close).clip(lower=0)
    avg_gain = gain.rolling(rsi_window).mean()
    avg_loss = loss.rolling(rsi_window).mean()
    rsi = 100 - 100 / (1 + avg_gain / avg_loss)
    rsi = rsi.fillna(100)  # if avg_loss is zero
    df[f'rsi_{rsi_window}'] = rsi

    # Range features
    df['range_1m'] = df['high'] - df['low']
    rolling_high = df['high'].rolling(range_window).max()
    rolling_low = df['low'].rolling(range_window).min()
    df[f'range_{range_window}'] = rolling_high - rolling_low

    # Volume and open interest changes
    df['volume_change'] = df['volume'].diff()
    df['oi_change'] = df['open_interest'].diff()

    # Spread to VWAP-like avg
    df['spread_vwap'] = df['close'] - df['avg']

    # Additional features
    df['abs_r_t'] = df['r_t'].abs()
    df['r_times_volume'] = df['r_t'] * df['volume']
    df['co_move'] = df['close'] - df['open']

    return df

if not data_raw.empty:
    data_feat = add_feature_columns(data_raw)
else:
    data_feat = pd.DataFrame()

print('Feature columns added. Current columns:')
print(list(data_feat.columns)[:15])


## Label generation

Creates the three-class label with threshold `alpha`. Labels align `X_t` with `Y_{t+1}` by shifting the close price by one minute. Rows with missing features or labels are dropped together.

In [ ]:
def generate_labels(df: pd.DataFrame, alpha: float = 0.001) -> pd.DataFrame:
    '''Generate direction labels using the provided alpha threshold.'''
    df = df.copy()
    close_next = df['close'].shift(-1)
    close_now = df['close']
    ratio = close_next / close_now
    conditions = [ratio > 1 + alpha, ratio < 1 - alpha]
    choices = [1, -1]
    df['label'] = np.select(conditions, choices, default=0)
    return df

if not data_feat.empty:
    data_labeled = generate_labels(data_feat, alpha=0.001)
    # Drop rows with any missing values from feature construction or the shifted label
    data_labeled = data_labeled.dropna().reset_index(drop=True)
else:
    data_labeled = pd.DataFrame()

print(f'Dataset after labeling and dropna: {data_labeled.shape}')
if not data_labeled.empty:
    print(data_labeled[['index', 'close', 'label']].head())


In [ ]:
# Inspect distinct months for raw and labeled data to understand coverage
if not data_raw.empty:
    months_raw = data_raw['index'].dt.to_period('M').unique()
    print('data_raw distinct months:', len(months_raw))
    print(months_raw)
else:
    print('data_raw is empty; no months to show.')

if not data_labeled.empty:
    months_labeled = data_labeled['index'].dt.to_period('M').unique()
    print('data_labeled distinct months:', len(months_labeled))
    print(months_labeled)
else:
    print('data_labeled is empty; no months to show.')



## Feature selection and scaling utilities

Selects the baseline feature set and standardizes using training statistics only. Scaling parameters are reused for the corresponding test fold to avoid leakage.

In [ ]:
BASELINE_FEATURES = [
    'r_t',
    'r_t_minus_1', 'r_t_minus_2', 'r_t_minus_3', 'r_t_minus_4', 'r_t_minus_5',
    'ma_5',
    'rsi_5',
    'range_5',
    'abs_r_t',
    'r_times_volume',
    'co_move',
]


def get_feature_matrix(df: pd.DataFrame, feature_list: List[str]) -> np.ndarray:
    missing = [f for f in feature_list if f not in df.columns]
    if missing:
        raise KeyError(f'Missing features: {missing}')
    return df[feature_list].values


def scale_train_test(X_train: np.ndarray, X_test: np.ndarray) -> Tuple[np.ndarray, np.ndarray, StandardScaler]:
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled, scaler


## Rolling-window split generator

Splits data in chronological order: 12 months for training and 1 month for testing by default. Adjust `train_months` and `test_months` as needed.

In [ ]:
def month_period(date_series: pd.Series) -> pd.Series:
    return date_series.dt.to_period('M')


def rolling_month_windows(df: pd.DataFrame, train_months: int = 12, test_months: int = 1):
    periods = month_period(df['index'])
    unique_months = pd.PeriodIndex(periods.unique()).sort_values()
    # Sorting via PeriodIndex avoids AttributeError on PeriodArray.unique() results
    start = 0
    while start + train_months + test_months <= len(unique_months):
        train_month_list = unique_months[start : start + train_months]
        test_month_list = unique_months[start + train_months : start + train_months + test_months]
        train_mask = periods.isin(train_month_list)
        test_mask = periods.isin(test_month_list)
        yield df[train_mask], df[test_mask], train_month_list, test_month_list
        start += test_months


## Model training and evaluation

Uses class-weighted multinomial logistic regression with L2 regularization. Performance is reported for each rolling window and averaged.

In [ ]:
LABEL_ORDER = [-1, 0, 1]


def compute_precisions(y_true: np.ndarray, y_pred: np.ndarray) -> Tuple[float, float]:
    """Compute precision for label 1 and label -1 with safe zero division handling."""
    prec_pos = precision_score(y_true, y_pred, labels=[1], average='macro', zero_division=0)
    prec_neg = precision_score(y_true, y_pred, labels=[-1], average='macro', zero_division=0)
    return prec_pos, prec_neg


def compute_vif_matrix(X: np.ndarray, feature_names: List[str]) -> pd.DataFrame:
    """Compute Variance Inflation Factor (VIF) for each feature using linear regression on others."""
    vif_rows = []
    X_with_intercept = np.column_stack([np.ones(X.shape[0]), X])
    for i in range(1, X_with_intercept.shape[1]):
        target = X_with_intercept[:, i]
        predictors = np.delete(X_with_intercept, i, axis=1)
        beta, _, _, _ = np.linalg.lstsq(predictors, target, rcond=None)
        fitted = predictors @ beta
        residuals = target - fitted
        ss_res = np.sum(residuals ** 2)
        ss_tot = np.sum((target - target.mean()) ** 2)
        r_squared = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
        vif = 1.0 / (1.0 - r_squared) if r_squared < 1 else np.inf
        vif_rows.append({'feature': feature_names[i - 1], 'vif': vif})
    return pd.DataFrame(vif_rows)


def save_table(df: pd.DataFrame, name: str) -> Path:
    """Save a DataFrame to the metrics directory with timestamped naming."""
    path = METRIC_DIR / f"{name}_{RUN_TIMESTAMP}.csv"
    df.to_csv(path, index=False)
    print(f"Saved table to {path}")
    return path


def train_and_evaluate(df: pd.DataFrame, C_values: List[float]) -> Dict[str, object]:
    """Run rolling-window evaluation across candidate C values and persist metrics/plots."""
    if df.empty:
        print('No data available for training.')
        return {}

    summary_records = []
    per_c_results: Dict[float, Dict[str, object]] = {}

    for C in C_values:
        fold_metrics = []
        fold_predictions = []
        fold_confusions = []
        model_last = None
        scaler_last = None
        X_train_scaled_last = None

        print(f"[Info] Evaluating C={C}")
        for fold_idx, (train_df, test_df, train_months, test_months) in enumerate(rolling_month_windows(df), start=1):
            print(f"  Fold {fold_idx}: train {train_months[0]} to {train_months[-1]}, test {test_months[0]}")
            X_train = get_feature_matrix(train_df, BASELINE_FEATURES)
            y_train = train_df['label'].values
            X_test = get_feature_matrix(test_df, BASELINE_FEATURES)
            y_test = test_df['label'].values

            # Scale using training statistics only
            X_train_scaled, X_test_scaled, scaler = scale_train_test(X_train, X_test)

            # Class weights inversely proportional to class frequency
            classes, counts = np.unique(y_train, return_counts=True)
            class_weights = {cls: 1.0 / cnt for cls, cnt in zip(classes, counts)}

            model = LogisticRegression(
                multi_class='multinomial',
                solver='lbfgs',
                max_iter=200,
                C=C,
                class_weight=class_weights,
                n_jobs=-1,
            )
            model.fit(X_train_scaled, y_train)
            preds = model.predict(X_test_scaled)
            proba = model.predict_proba(X_test_scaled)

            acc = accuracy_score(y_test, preds)
            f1 = f1_score(y_test, preds, average='macro')
            prec_pos, prec_neg = compute_precisions(y_test, preds)
            start_time = test_df['index'].min()
            end_time = test_df['index'].max()
            fold_metrics.append({
                'C': C,
                'fold': fold_idx,
                'start_time': start_time,
                'end_time': end_time,
                'accuracy': acc,
                'f1_macro': f1,
                'precision_1': prec_pos,
                'precision_minus1': prec_neg,
            })
            print(f"    Fold {fold_idx} metrics -> accuracy: {acc:.4f}, macro F1: {f1:.4f}, precision@1: {prec_pos:.4f}, precision@-1: {prec_neg:.4f}")

            # Store predictions with probabilities
            prob_cols = {}
            for cls_idx, cls_label in enumerate(model.classes_):
                prob_cols[f'prob_{int(cls_label)}'] = proba[:, cls_idx]
            fold_pred_df = pd.DataFrame({
                'fold': fold_idx,
                'timestamp': test_df['index'].values,
                'true_label': y_test,
                'pred_label': preds,
                **prob_cols,
            })
            fold_predictions.append(fold_pred_df)

            cm = confusion_matrix(y_test, preds, labels=LABEL_ORDER)
            fold_confusions.append(cm)

            # Keep last model/scaler for downstream diagnostics
            model_last = model
            scaler_last = scaler
            X_train_scaled_last = X_train_scaled

        if fold_metrics:
            avg_acc = np.mean([fs['accuracy'] for fs in fold_metrics])
            avg_f1 = np.mean([fs['f1_macro'] for fs in fold_metrics])
            summary_records.append({'C': C, 'avg_accuracy': avg_acc, 'avg_f1_macro': avg_f1})
            per_c_results[C] = {
                'fold_metrics': fold_metrics,
                'predictions': fold_predictions,
                'confusions': fold_confusions,
                'model': model_last,
                'scaler': scaler_last,
                'X_train_scaled': X_train_scaled_last,
            }
            print(f"[Summary] C={C} -> avg accuracy: {avg_acc:.4f}, avg macro F1: {avg_f1:.4f}")

    summary_df = pd.DataFrame(summary_records)
    if summary_df.empty:
        print('No summary metrics were produced.')
        return {}

    # Select best C by macro F1 then accuracy
    summary_sorted = summary_df.sort_values(['avg_f1_macro', 'avg_accuracy'], ascending=[False, False]).reset_index(drop=True)
    best_C = summary_sorted.loc[0, 'C']
    print(f"Best C selected by macro F1 then accuracy: {best_C}")
    print(summary_sorted)

    best_details = per_c_results[best_C]
    rolling_df = pd.DataFrame(best_details['fold_metrics'])
    rolling_path = save_table(rolling_df, 'rolling_metrics')
    if not rolling_df.empty:
        acc_mean, acc_std = rolling_df['accuracy'].mean(), rolling_df['accuracy'].std(ddof=0)
        f1_mean, f1_std = rolling_df['f1_macro'].mean(), rolling_df['f1_macro'].std(ddof=0)
        print(f"[Rolling] Accuracy mean={acc_mean:.4f}, std={acc_std:.4f}; Macro F1 mean={f1_mean:.4f}, std={f1_std:.4f}")

    preds_df = pd.concat(best_details['predictions'], ignore_index=True)
    preds_path = save_table(preds_df, 'predictions_detailed')

    # Aggregate confusion matrices across folds
    total_conf = np.sum(best_details['confusions'], axis=0)
    conf_df = pd.DataFrame(total_conf, index=LABEL_ORDER[:total_conf.shape[0]], columns=LABEL_ORDER[:total_conf.shape[1]])
    conf_path = METRIC_DIR / f"confusion_matrix_{RUN_TIMESTAMP}.csv"
    conf_df.to_csv(conf_path)
    print(f"Saved aggregated confusion matrix to {conf_path}")
    fp = conf_df.loc[-1, 1] if 1 in conf_df.columns else 0
    fn = conf_df.loc[1, -1] if -1 in conf_df.columns else 0
    print(f"Total false positives (true -1 predicted 1): {fp}")
    print(f"Total false negatives (true 1 predicted -1): {fn}")

    # Logistic coefficients and parameters
    coeff_rows = []
    for class_idx, cls in enumerate(best_details['model'].classes_):
        coeff_rows.append({'class': int(cls), 'feature': 'intercept', 'coefficient': best_details['model'].intercept_[class_idx]})
        for fname, coef_val in zip(BASELINE_FEATURES, best_details['model'].coef_[class_idx]):
            coeff_rows.append({'class': int(cls), 'feature': fname, 'coefficient': coef_val})
    coeff_df = pd.DataFrame(coeff_rows)
    coeff_path = save_table(coeff_df, 'logit_coefficients')

    top_coeff = (
        coeff_df[coeff_df['feature'] != 'intercept']
        .assign(abs_coef=lambda d: d['coefficient'].abs())
        .sort_values('abs_coef', ascending=False)
        .groupby('class')
        .head(5)
    )
    print('Top absolute coefficients by class:')
    print(top_coeff[['class', 'feature', 'coefficient']])
    intercept_info = coeff_df[coeff_df['feature'] == 'intercept'][['class', 'coefficient']]
    print('Intercepts by class:')
    print(intercept_info)

    param_rows = []
    for feat_idx, (fname, mean_val, scale_val) in enumerate(zip(BASELINE_FEATURES, best_details['scaler'].mean_, best_details['scaler'].scale_)):
        row = {'feature': fname, 'mean': mean_val, 'scale': scale_val}
        for class_idx, cls in enumerate(best_details['model'].classes_):
            row[f'coef_class_{int(cls)}'] = best_details['model'].coef_[class_idx][feat_idx]
        param_rows.append(row)
    intercept_row = {'feature': 'intercept', 'mean': np.nan, 'scale': np.nan}
    for class_idx, cls in enumerate(best_details['model'].classes_):
        intercept_row[f'coef_class_{int(cls)}'] = best_details['model'].intercept_[class_idx]
    param_rows.append(intercept_row)
    param_df = pd.DataFrame(param_rows)
    param_path = save_table(param_df, 'logit_params')

    # VIF calculation on the last training fold
    vif_df = compute_vif_matrix(best_details['X_train_scaled'], BASELINE_FEATURES)
    vif_path = save_table(vif_df, 'vif_summary')
    if not vif_df.empty:
        high_vif = vif_df[vif_df['vif'] > 10].sort_values('vif', ascending=False)
        if not high_vif.empty:
            print('Features with VIF > 10 (potential multicollinearity):')
            print(high_vif)
        top_vif = vif_df.sort_values('vif', ascending=False).head()
        print('Top VIF features:')
        print(top_vif)

    # Calibration curve for class 1 probability
    prob_col = 'prob_1'
    if prob_col in preds_df.columns:
        bins = np.linspace(0, 1, 11)
        calib_rows = []
        for i in range(len(bins) - 1):
            mask = (preds_df[prob_col] >= bins[i]) & (preds_df[prob_col] < bins[i + 1] if i < len(bins) - 2 else preds_df[prob_col] <= bins[i + 1])
            bin_data = preds_df[mask]
            if bin_data.empty:
                continue
            true_rate = (bin_data['true_label'] == 1).mean()
            calib_rows.append({'bin_left': bins[i], 'bin_right': bins[i + 1], 'bin_center': (bins[i] + bins[i + 1]) / 2, 'true_rate': true_rate})
        calib_df = pd.DataFrame(calib_rows)
        calib_path = save_table(calib_df, 'calibration_curve')
    else:
        calib_df = pd.DataFrame()
        calib_path = None

    results = {
        'summary': summary_sorted,
        'best_C': best_C,
        'rolling_metrics': rolling_df,
        'predictions': preds_df,
        'confusion': conf_df,
        'coefficients': coeff_df,
        'logit_params': param_df,
        'vif': vif_df,
        'calibration': calib_df,
        'paths': {
            'rolling_metrics': rolling_path,
            'predictions': preds_path,
            'confusion': conf_path,
            'coefficients': coeff_path,
            'params': param_path,
            'vif': vif_path,
            'calibration': calib_path,
        },
    }
    return results

In [ ]:
def save_and_show(fig, name: str):
    path = FIG_DIR / f"{name}_{RUN_TIMESTAMP}.png"
    fig.savefig(path, bbox_inches='tight', dpi=150)
    plt.show()
    print(f"Saved figure to {path}")


def plot_price_volume(df: pd.DataFrame, title: str = 'Price and Volume Overview'):
    fig, ax1 = plt.subplots(figsize=(12, 6))
    ax1.plot(df['index'], df['close'], color='steelblue', label='Close')
    ax1.set_ylabel('Close Price')
    ax1.set_title(title)
    ax2 = ax1.twinx()
    ax2.bar(df['index'], df['volume'], color='darkorange', alpha=0.3, label='Volume')
    ax2.set_ylabel('Volume')
    ax1.legend(loc='upper left')
    ax2.legend(loc='upper right')
    save_and_show(fig, 'price_volume_overview')


def plot_class_distribution(df: pd.DataFrame, title: str = 'Label Distribution'):
    counts = df['label'].value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(6, 4))
    sns.barplot(x=counts.index.astype(int), y=counts.values, palette='Blues', ax=ax)
    ax.set_xlabel('Label')
    ax.set_ylabel('Count')
    ax.set_title(title)
    save_and_show(fig, 'label_distribution')


def plot_feature_by_label(df: pd.DataFrame, feature: str, title: Optional[str] = None):
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.boxplot(data=df, x='label', y=feature, palette='Set2', ax=ax)
    ax.set_title(title or f'{feature} by label')
    save_and_show(fig, f'feature_{feature}_by_label')


def plot_correlation_heatmap(df: pd.DataFrame, feature_list: List[str]):
    corr = df[feature_list].corr()
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(corr, cmap='coolwarm', annot=False, ax=ax)
    ax.set_title('Feature Correlation Heatmap')
    save_and_show(fig, 'feature_correlation_heatmap')


def plot_probability_distribution(pred_df: pd.DataFrame, prob_column: str = 'prob_1', bins: int = 20):
    if pred_df.empty or prob_column not in pred_df.columns:
        print(f'Cannot plot probability distribution; column {prob_column} missing.')
        return
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(pred_df[prob_column], bins=bins, color='teal', alpha=0.7, edgecolor='black')
    ax.set_xlabel(f'Predicted probability for class {prob_column.replace("prob_", "")}')
    ax.set_ylabel('Frequency')
    ax.set_title('Prediction Probability Distribution')
    save_and_show(fig, 'probability_hist')


def plot_rolling_performance(rolling_df: pd.DataFrame):
    if rolling_df.empty:
        print('No rolling metrics to plot.')
        return
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(rolling_df['start_time'], rolling_df['accuracy'], marker='o', label='Accuracy')
    ax.plot(rolling_df['start_time'], rolling_df['f1_macro'], marker='o', label='Macro F1')
    ax.set_xlabel('Test window start')
    ax.set_ylabel('Score')
    ax.set_title('Rolling Window Performance')
    ax.legend()
    save_and_show(fig, 'rolling_performance')


def plot_confusion_heatmap(conf_df: pd.DataFrame, title: str = 'Confusion Matrix'):
    if conf_df.empty:
        print('No confusion matrix to plot.')
        return
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(conf_df, annot=True, fmt='g', cmap='Blues', ax=ax)
    ax.set_xlabel('Predicted label')
    ax.set_ylabel('True label')
    ax.set_title(title)
    save_and_show(fig, 'confusion_matrix')


def plot_logit_coefficients(coeff_df: pd.DataFrame, target_class: int = 1, top_n: int = 15):
    if coeff_df.empty:
        print('No coefficients to plot.')
        return
    subset = coeff_df[(coeff_df['class'] == target_class) & (coeff_df['feature'] != 'intercept')]
    subset = subset.assign(abs_coef=subset['coefficient'].abs()).sort_values('abs_coef', ascending=False).head(top_n)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.barplot(data=subset, x='coefficient', y='feature', palette='coolwarm', ax=ax)
    ax.set_title(f'Logistic Coefficients (class {target_class})')
    save_and_show(fig, 'logit_coef_barplot')


def plot_calibration_curve_from_df(calib_df: pd.DataFrame):
    if calib_df.empty:
        print('No calibration data to plot.')
        return
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot(calib_df['bin_center'], calib_df['true_rate'], marker='o', label='Observed')
    ax.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfect calibration')
    ax.set_xlabel('Predicted probability bin center (class 1)')
    ax.set_ylabel('Observed positive rate')
    ax.set_title('Calibration Plot')
    ax.legend()
    save_and_show(fig, 'calibration_plot')



In [ ]:
def run_full_pipeline(data_raw: pd.DataFrame,
                      data_labeled: pd.DataFrame,
                      csv_files: List[Path],
                      C_values: List[float],
                      feature_list: List[str],
                      boxplot_features: Optional[List[str]] = None):
    """
    High-level pipeline:
    - Construct contract name string from csv file names
    - Train the model once and select the best C
    - Generate all visualizations (add contract name in titles when possible)
    """
    if data_labeled.empty:
        print('data_labeled is empty, nothing to train/plot.')
        return {}

    # === 1. Contract name string ===
    contract_names = [p.stem for p in csv_files]  # e.g., "AG_主力合约_1m数据"
    if len(contract_names) == 1:
        contract_title = contract_names[0]
    else:
        # Multiple contracts: join with "&" for display purposes
        contract_title = ' & '.join(contract_names)

    print(f'Training on contracts: {contract_title}')
    print(f'Number of labeled rows: {len(data_labeled)}')

    # === 2. Train the model once (best C selected internally) ===
    results = train_and_evaluate(data_labeled, C_values)
    if not results:
        print('No results returned from train_and_evaluate.')
        return results

    # === 3. EDA visualizations ===
    print('=== Plotting EDA figures ===')

    # 3.1 Price & volume plot (include contract name in title)
    plot_price_volume(
        data_raw,
        title=f'{contract_title} Price & Volume'
    )

    # 3.2 Class distribution plot
    plot_class_distribution(
        data_labeled,
        title=f'{contract_title} Label Distribution'
    )

    # 3.3 Feature distributions by label (boxplots); default first few features
    if boxplot_features is None:
        boxplot_features = feature_list[:5]

    for feat in boxplot_features:
        if feat in data_labeled.columns:
            plot_feature_by_label(
                data_labeled,
                feature=feat,
                title=f'{contract_title}: {feat} by label'
            )

    # 3.4 Feature correlation heatmap (title fixed inside the function)
    plot_correlation_heatmap(data_labeled, feature_list=feature_list)

    # === 4. Model diagnostic plots (based on best C) ===
    print('=== Plotting model diagnostics for best C ===')
    print('Best C:', results['best_C'])

    # 4.1 Probability distribution of predictions
    plot_probability_distribution(results['predictions'])

    # 4.2 Rolling window performance curves
    plot_rolling_performance(results['rolling_metrics'])

    # 4.3 Confusion matrix heatmap (add contract name in title)
    plot_confusion_heatmap(
        results['confusion'],
        title=f'{contract_title} Confusion Matrix'
    )

    # 4.4 Logistic regression coefficients for class=1
    plot_logit_coefficients(results['coefficients'], target_class=1, top_n=15)

    # 4.5 Calibration curve
    plot_calibration_curve_from_df(results['calibration'])

    return results



In [ ]:
if not data_labeled.empty:
    candidate_C = [0.0001, 0.001, 0.01, 0.1, 1.0]
    results = run_full_pipeline(
        data_raw=data_raw,
        data_labeled=data_labeled,
        csv_files=csv_files,
        C_values=candidate_C,
        feature_list=BASELINE_FEATURES,
        boxplot_features=BASELINE_FEATURES[:5],  
    )
else:
    print('data_labeled is empty, skip training and plotting.')
    results = {}



## Quick sanity run (optional)

Uncomment the following cell to run a light evaluation over a few folds and C values. Keep `MAX_FILES`, `NROWS_PER_FILE`, and the date filters small to avoid long runtimes. This is intended for exploratory checks inside the notebook.


## Visualization helpers and saved outputs

Figures and metrics are written into a run-specific folder under `outputs/<timestamp>/`. Titles, labels, and legends remain in English. Use the helpers to generate PNG figures plus CSV tables for diagnostics: probability distribution, rolling performance curve, confusion matrix heatmap, coefficient bars, and calibration plot. Key tables also capture rolling means/std, detailed predictions, aggregated confusion counts, combined scaler + coefficient parameters, and VIF summaries (with high-VIF warnings printed in the notebook).


## Next steps

1. Increase `MAX_FILES`, `NROWS_PER_FILE`, and widen the date range when ready to process more data.
2. Uncomment the evaluation cell to run rolling-window experiments.
3. Generate the recommended visualizations using the helper functions above. All figures are saved as PNG under `outputs/<timestamp>/figures`.
4. Consider extending the feature set or testing linear SVM by swapping the model inside `train_and_evaluate`.
